In [9]:
import getpass
from datetime import datetime

from pptx import Presentation


def replace_pptx_content(input_path: str, text_data: dict, image_data: dict, output_path: str):
    # 加载 PPT
    prs = Presentation(input_path)

    for slide in prs.slides:
        # 使用 list(slide.shapes) 因为我们在循环中涉及删除操作
        for shape in list(slide.shapes):

            # 仅处理包含文本框的形状
            if not shape.has_text_frame:
                continue

            # --- 逻辑 1：处理图片替换 ---
            # 先检查是否匹配图片关键词，因为图片替换后会删除该文本框
            replaced_with_image = False
            for key, img_path in image_data.items():
                if key in shape.text_frame.text:
                    # 获取原位置坐标
                    left, top, width, height = shape.left, shape.top, shape.width, shape.height
                    # 插入图片（保持宽度，高度自适应）
                    slide.shapes.add_picture(img_path, left, top, width=width)
                    # 从幻灯片中移除该占位文本框
                    sp = shape._element
                    sp.getparent().remove(sp)
                    replaced_with_image = True
                    break  # 已替换为图片，跳出当前 shape 的处理

            if replaced_with_image:
                continue

            # --- 逻辑 2：处理普通文本替换 ---
            for paragraph in shape.text_frame.paragraphs:
                for run in paragraph.runs:
                    for key, value in text_data.items():
                        if key in run.text:
                            run.text = run.text.replace(key, value)

    # 保存文件
    prs.save(output_path)
    print(f"PPT 已成功保存至: {output_path}")


# --- 数据准备 ---
current_user = getpass.getuser()
current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# 文本替换字典
text_dict = {
    "{{title}}": "季度销售汇报",
    "{{name}}": current_user,
    "{{time}}": current_time
}

# 图片替换字典 (Key 是 PPT 里的占位文本，Value 是本地图片路径)
image_dict = {
    "{{logo}}": "/Users/administrator/Desktop/pptx/1.png"
}

# 文件路径
input_ppt = '/Users/administrator/Desktop/pptx/template.pptx'
output_ppt = '/Users/administrator/Desktop/pptx/output.pptx'

# --- 执行 ---
replace_pptx_content(
    input_path=input_ppt,
    text_data=text_dict,
    image_data=image_dict,
    output_path=output_ppt
)

PPT 已成功保存至: /Users/administrator/Desktop/pptx/output.pptx
